In [1]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load PDF (keep file in same folder)
reader = PdfReader("indian-penal-code.pdf")

# Extract text
text = ""
for p in reader.pages:
    t = p.extract_text()
    if t:
        text += t

text = text.replace("\n", " ")

# Split into sections
docs = ["Section " + d.strip()
        for d in text.split("Section")
        if len(d.strip()) > 30] 

print("Sections:", len(docs))

# Embedding model
model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(docs)

# Chatbot function
def chatbot(q):
    m = re.search(r'section\s*(\d+)', q.lower())

    if m:
        for d in docs:
            if f"Section {m.group(1)}" in d:
                return d
        return "Section not found."

    q_emb = model.encode([q])
    sim = cosine_similarity(q_emb, embeddings)[0]
    return docs[sim.argmax()]

# CLI loop
while True:
    q = input("You: ")
    if q.lower() == "exit":
        break
    print("Bot:", chatbot(q), "\n")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Sections: 616
You: explain meaninf of word judge in 10 words
Bot: Section 19. Judge “Judge”.–The word “Judge” denotes not only every person who is officially  designated as a Judge, but also every person who is empowered by law to give, in any legal proceeding, civil or criminal, a definitive judgment, or a judgment  which, if not appealed against, would be definitive, or a judgment which, if  confirmed by some other authority, would be definitive, or who is one of a body of persons, which body of persons is empowered by law  to give such a judgment. Illustrations (a) A Collector exercising jurisdiction in a suit under Act 10 of 1859, is a judge. (b) A Magistrate exercising jurisdiction in respect of a charge on which he has  power to sentence to fine or imprisonment, with or without appeal, is a judge. (c) A member of a Panchayat which has power, under 1Regulation VII, 1816, of the Madras Code, to try and determine suits, is a judge. (d) A Magistrate exercising jurisdiction in respect

In [ ]:
import os
import re
import pickle
import numpy as np
import ollama
from pypdf import PdfReader

PDF_PATH = "indian-penal-code.pdf"
MODEL = "nomic-embed-text"
CACHE_FILE = "ipc_cache.pkl"
BATCH_SIZE = 64

def read_pdf(path):
    reader = PdfReader(path)
    return " ".join((p.extract_text() or "") for p in reader.pages).replace("\n", " ")

def split_sections(text):
    docs = [
        m.group(0).strip()
        for m in re.finditer(r"(Section\s+\d+\b.*?)(?=Section\s+\d+\b|$)", text, re.I | re.S)
    ]
    return docs if docs else [text.strip()]

def embed(texts):
    resp = ollama.embed(model=MODEL, input=texts)
    vecs = resp["embeddings"] if isinstance(resp, dict) else resp.embeddings
    return np.array(vecs, dtype=np.float32)

def normalize(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)

def build_index():
    text = read_pdf(PDF_PATH)
    docs = split_sections(text)

    vec_parts = []
    for i in range(0, len(docs), BATCH_SIZE):
        vec_parts.append(embed(docs[i:i + BATCH_SIZE]))

    vecs = normalize(np.vstack(vec_parts))
    return docs, vecs

def load_index():
    mtime = os.path.getmtime(PDF_PATH)

    if os.path.exists(CACHE_FILE):
        try:
            data = pickle.load(open(CACHE_FILE, "rb"))
            if data["model"] == MODEL and data["mtime"] == mtime:
                return data["docs"], data["vecs"]
        except Exception:
            pass

    docs, vecs = build_index()
    pickle.dump(
        {"model": MODEL, "mtime": mtime, "docs": docs, "vecs": vecs},
        open(CACHE_FILE, "wb"),
    )
    return docs, vecs

docs, vecs = load_index()

def chatbot(q):
    m = re.search(r"section\s*(\d+)", q, re.I)
    if m:
        target = f"Section {m.group(1)}"
        for d in docs:
            if re.match(rf"^{re.escape(target)}\b", d, re.I):
                return d
        return "Section not found."

    q_vec = normalize(embed([q]))[0]
    idx = int(np.argmax(vecs @ q_vec))
    return docs[idx]

while True:
    q = input("You: ").strip()
    if q.lower() == "exit":
        break
    print("Bot:", chatbot(q), "\n")